In [4]:
import os

# 1. Download the zip file directly using the File ID
file_id = "1qsG1H7LNQKBVU_6UO9jwjujvP0npkdOA"
print("Downloading Shapes.zip directly to Colab...")
!gdown {file_id} -O /content/Shapes.zip

# 2. Unzip it
print("\nUnzipping the dataset...")
!unzip -q /content/Shapes.zip -d /content/

# 3. Verify the folders
print("\nChecking folders:")
base_path = "/content/Shapes"
if not os.path.exists(base_path):
    base_path = "/content/Shapes/Shapes" # fallback

for folder in os.listdir(base_path):
    path = os.path.join(base_path, folder)
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith((".png", ".jpg"))])
        print(f"  {folder}: {count} images")

Downloading...
From (original): https://drive.google.com/uc?id=1qsG1H7LNQKBVU_6UO9jwjujvP0npkdOA
From (redirected): https://drive.google.com/uc?id=1qsG1H7LNQKBVU_6UO9jwjujvP0npkdOA&confirm=t&uuid=717abdf0-7e2c-4226-93df-f16fb240c663
To: /content/Shapes.zip
100% 900M/900M [00:12<00:00, 72.3MB/s]

Unzipping the dataset...

Checking folders:
  Diagonal: 554 images
  Overlapped pencils: 445 images
  Circle: 575 images
  Overlapped circle: 534 images
  Triangle: 580 images
  Square: 584 images
  Star: 477 images
  Wave: 550 images


In [5]:
import os

bad_images = [
    "img1141-W-4.png", "img1182-W-4.png", "img4051-W-4.png", "img4293-W-4.png",
    "img1955-W-3(1011).png", "img3930-W-3(1011).png",
    "img2489-T-5.png", "img3000-T-5.png", "img4294-T-5.png", "img2251-T-5.png",
    "img2302-T-0.png", "img1183-T-5.png", "img1706-T-5.png",
    "img3164-ST-3(11001).png", "img3054-ST-4(11101).png",
    "img1421-D-5.png", "img1549-D-5.png", "img1957-D-5.png",
    "img2303-D-0.png", "img2305-D-4(11011).png", "img4067-D-5.png",
    "img3053-D-5.png", "img2185-D-0.png",
    "img1954-O-5(101111).png", "img4292-O-6.png"
]

deleted_count = 0
for root, dirs, files in os.walk("/content/Shapes"):
    for file in files:
        if file in bad_images:
            os.remove(os.path.join(root, file))
            deleted_count += 1
print(f"Deleted: {deleted_count} images.")

Deleted: 25 images.


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_dir = "/content/Shapes"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)

train_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

Found 3422 images belonging to 8 classes.
Found 852 images belonging to 8 classes.


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

print("Building the Deeper Scoring CNN (4 Conv Blocks)...")

# 1. Build the Deeper CNN
model_deep = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Block 4
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Classification Head
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

# 2. Compile
model_deep.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_deep.summary()

# 3. Set up saving to Google Drive
save_path = "/content/drive/MyDrive/Shapes/deep_scoring_cnn.keras"
checkpoint = ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True, verbose=1)

# 4. Train for 15 Epochs (The GPU will make this fast)
print("\nTraining the Deeper Scoring CNN...")
history = model_deep.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True), checkpoint]
)

print("\nTraining Complete! Model saved to Drive.")

Building the Deeper Scoring CNN (4 Conv Blocks)...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,296 (8.93 MB)

 Trainable params: 2,340,296 (8.93 MB)

 Non-trainable params: 0 (0.00 B)


Training the Deeper Scoring CNN...
Epoch 1/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 592ms/step - accuracy: 0.1542 - loss: 2.0867
Epoch 1: val_accuracy improved from None to 0.53521, saving model to /content/drive/MyDrive/Shapes/deep_scoring_cnn.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Shapes/deep_scoring_cnn.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 97s 790ms/step - accuracy: 0.2107 - loss: 1.9888 - val_accuracy: 0.5352 - val_loss: 1.2979
Epoch 2/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 506ms/step - accuracy: 0.6782 - loss: 0.9427
Epoch 2: val_accuracy improved from 0.53521 to 0.90258, saving model to /content/drive/MyDrive/Shapes/deep_scoring_cnn.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Shapes/deep_scoring_cnn.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 69s 643ms/step - accuracy: 0.7858 - loss: 0.6863 - val_accuracy: 0.9026 - val_loss: 0.4122
Epoch 3/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - accuracy: 0.9098 - loss: 0.3579
Epoch 3: val_accuracy improved from 0

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
import os
import random

print("Loading the trained deeper CNN...")
# Load the model we just saved to Drive
model = tf.keras.models.load_model("/content/drive/MyDrive/Shapes/deep_scoring_cnn.keras")

# The classes must match the order of the folders
class_names = ['Circle', 'Diagonal', 'Overlapped circle', 'Overlapped pencils',
               'Square', 'Star', 'Triangle', 'Wave']

# Pick a random image from the Circle folder
test_folder = "/content/Shapes/Circle"
random_image_name = random.choice(os.listdir(test_folder))
test_image_path = os.path.join(test_folder, random_image_name)

print(f"\nTesting on image: {random_image_name}")
print("-" * 40)

# Load and preprocess the image
img = keras_image.load_img(test_image_path, target_size=(128, 128))
img_array = keras_image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Predict
preds = model.predict(img_array, verbose=0)
predicted_class_idx = np.argmax(preds)
predicted_class = class_names[predicted_class_idx]
confidence = preds[0][predicted_class_idx] * 100

print(f"Predicted Shape: {predicted_class} (Confidence: {confidence:.2f}%)")

# Extract the actual true score from the filename (e.g., img1003-C-3.png -> 3)
try:
    true_score = int(random_image_name.split('-')[-1].split('(')[0].split('.')[0])
    print(f"Actual True Score in Filename: {true_score}")
except:
    print("Could not extract true score from filename.")


Loading the trained deeper CNN...

Testing on image: img3103-C-3(1011).png
----------------------------------------
Predicted Shape: Circle (Confidence: 100.00%)
Actual True Score in Filename: 3
